# Feature builder — imputed parquet → model inputs

Last transform before modelling, and the last place a train/test boundary can be
violated. For each **fit set × imputation** this notebook fits one `FeatureSpec` on
training rows only, applies it to that fit set's own rows and its held-out rows, and
writes both representations plus the spec as JSON.

| Fitted on | Applied to |
|---|---|
| `dev_fold{f}_train` | `dev_fold{f}_train`, `dev_fold{f}_test` |
| `dev_full` | `dev_full`, `temporal`, `exploratory` |
| `pretrain` (masked) | `pretrain` only |

Both representations — integer codes for the neural variants' embeddings, one-hot for
the classical baselines — come from the **same fitted spec**. That is what guarantees
every model receives identical information.

Requires: `pandas`, `pyarrow`, `numpy` (and `torch` only if you re-run the test suite).

## 1. Set your paths and the DIAG reference

In [1]:
# ==== EDIT THIS: folder holding the imputed parquet files ====
IMPUTED_DIR = r"C:\Users\olf3ph\OneDrive - University of Missouri\Documents\Research\GRAFT\CS5402\IMPUTATION\imputed"

# ==== EDIT THIS: where the transformed arrays and specs are written ====
OUTPUT_DIR = r"C:\Users\olf3ph\OneDrive - University of Missouri\Documents\Research\GRAFT\CS5402\FEATURES"

# ==== EDIT THIS: folder containing rp_core (so `features` can be imported) ====
RP_CORE_DIR = r"C:\Users\olf3ph\OneDrive - University of Missouri\Documents\Research\GRAFT\CS5402\MODEL CORE\rp_core (5)"

# ==== EDIT THIS after running the DIAG inspection cell below ====
# Code for the hypertensive/vascular DIAG group, which the analysis plan uses as the
# reference level. Leave as None to fall back to the modal level (valid, but not the
# intended contrast — the builder warns when it does this).
DIAG_REFERENCE = 2

N_IMPUTATIONS = 5
N_FOLDS = 5

In [2]:
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, RP_CORE_DIR)
from features import (CATEGORICAL, CONTINUOUS, PRETRAIN_MASKED, ROBUST_COLUMNS,
                      FeatureSpec, check_schema, encoder_config_kwargs,
                      fit_feature_spec, summarise, transform)

IMPUTED = Path(IMPUTED_DIR)
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

assert IMPUTED.exists(), f"IMPUTED_DIR not found: {IMPUTED}"
found = sorted(p.name for p in IMPUTED.glob('*.parquet'))
print(f"{len(found)} parquet files in {IMPUTED}")
print("  " + ", ".join(found[:6]) + (" ..." if len(found) > 6 else ""))

70 parquet files in C:\Users\olf3ph\OneDrive - University of Missouri\Documents\Research\GRAFT\CS5402\IMPUTATION\imputed
  dev_fold0_test_m1.parquet, dev_fold0_test_m2.parquet, dev_fold0_test_m3.parquet, dev_fold0_test_m4.parquet, dev_fold0_test_m5.parquet, dev_fold0_train_m1.parquet ...


## 2. Schema check

Run this before building anything: a mismatch should surface as a readable report,
not as a `KeyError` twenty files in. `BMI_CALC` / `BMI_DON_CALC` are expected to be
absent — they are derived from height and weight at build time, which is deterministic
arithmetic and introduces no fitted quantity.

In [3]:
probe = pd.read_parquet(IMPUTED / "dev_full_m1.parquet")
print(f"dev_full_m1: {len(probe):,} rows x {probe.shape[1]} columns\n")
rep = check_schema(probe)
for k, v in rep.items():
    print(f"  {k:24s} {v}")
print(f"\nexpected: {len(CONTINUOUS)} continuous + {len(CATEGORICAL)} categorical "
      f"= {len(CONTINUOUS) + len(CATEGORICAL)} features")

dev_full_m1: 47,290 rows x 59 columns

  missing_continuous       []
  missing_categorical      []
  missing_passthrough      []
  present_but_dropped      ['ETHCAT', 'ETHCAT_DON']
  unexpected               ['DWFG_KI', 'GSTATUS_KI', 'GTIME_KI', 'ev_cause1', 'ev_cause2', 'na_cumhaz_cause1', 'na_cumhaz_cause2', 'pretrain_split']

expected: 16 continuous + 27 categorical = 43 features


In [4]:
# DIAG levels with counts — read the hypertensive/vascular code off this table and
# set DIAG_REFERENCE in cell 1, then re-run from there.
print(probe['DIAG'].value_counts().sort_index().to_string())

DIAG
1     11290
2     12228
3      3313
4      3862
5      3042
6      2429
7      2792
8      1873
9      1879
10     1055
11     1501
12      938
13     1088


## 3. Build

One spec per (fit set × imputation). Reference levels are pinned from imputation 1
across the rest of a fit set: the modal level is a sample quantity and can flip
between imputations, which leaves predictions unchanged but makes pooled baseline
coefficients across the five imputations meaningless. Those are the same patients, so
pinning is one fitted choice applied consistently, not leakage.

In [5]:
def fit_sets():
    """(fit_name, apply_names, masked) — the fit/apply table above."""
    for f in range(N_FOLDS):
        yield (f"dev_fold{f}_train", [f"dev_fold{f}_train", f"dev_fold{f}_test"], False)
    yield ("dev_full", ["dev_full", "temporal", "exploratory"], False)
    yield ("pretrain", ["pretrain"], True)


def save(result, spec, name, m):
    """Arrays + column names for one applied set."""
    stem = OUT / f"{name}_m{m}"
    if result.representation == "embedding":
        np.savez_compressed(f"{stem}_embedding.npz",
                            continuous=result.continuous,
                            categorical_codes=result.categorical_codes,
                            columns=np.array(result.columns, dtype=object),
                            cardinalities=np.array(result.cardinalities))
    else:
        np.savez_compressed(f"{stem}_onehot.npz",
                            design=result.design,
                            columns=np.array(result.columns, dtype=object))
        # Outcome / administrative columns, written once per applied set.
        result.passthrough.to_parquet(f"{stem}_outcomes.parquet", index=False)


summary_rows, unseen_log = [], []

for fit_name, apply_names, masked in fit_sets():
    pinned = None
    for m in range(1, N_IMPUTATIONS + 1):
        train_df = pd.read_parquet(IMPUTED / f"{fit_name}_m{m}.parquet")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            spec = fit_feature_spec(train_df, fit_name=f"{fit_name}_m{m}",
                                    masked=masked, diag_reference=DIAG_REFERENCE,
                                    reference_overrides=pinned)
        if pinned is None:
            pinned = spec.categorical_reference      # pin for m = 2..N
        spec.to_json(OUT / f"spec_{fit_name}_m{m}.json")

        for apply_name in apply_names:
            df = (train_df if apply_name == fit_name
                  else pd.read_parquet(IMPUTED / f"{apply_name}_m{m}.parquet"))
            for rep in ("embedding", "onehot"):
                res = transform(df, spec, rep)
                save(res, spec, apply_name, m)
                if rep == "embedding":
                    summary_rows.append({
                        "fit_set": fit_name, "applied_to": apply_name, "m": m,
                        "n_rows": res.n_rows,
                        "n_embedding": spec.n_embedding_features,
                        "n_onehot": spec.n_onehot_features,
                        "unseen_total": int(sum(res.unseen_counts.values())),
                    })
                    for col, cnt in res.unseen_counts.items():
                        unseen_log.append({"fit_set": fit_name,
                                           "applied_to": apply_name, "m": m,
                                           "column": col, "n_rows": cnt,
                                           "values": res.unseen_values[col][:5]})
        print(f"  {fit_name}_m{m}: {spec.n_embedding_features} embedding / "
              f"{spec.n_onehot_features} one-hot features -> "
              f"{', '.join(apply_names)}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "build_summary.csv", index=False)
print(f"\nwrote {len(summary)} applied sets to {OUT}")

  dev_fold0_train_m1: 43 embedding / 104 one-hot features -> dev_fold0_train, dev_fold0_test
  dev_fold0_train_m2: 43 embedding / 104 one-hot features -> dev_fold0_train, dev_fold0_test
  dev_fold0_train_m3: 43 embedding / 104 one-hot features -> dev_fold0_train, dev_fold0_test
  dev_fold0_train_m4: 43 embedding / 104 one-hot features -> dev_fold0_train, dev_fold0_test
  dev_fold0_train_m5: 43 embedding / 104 one-hot features -> dev_fold0_train, dev_fold0_test
  dev_fold1_train_m1: 43 embedding / 104 one-hot features -> dev_fold1_train, dev_fold1_test
  dev_fold1_train_m2: 43 embedding / 104 one-hot features -> dev_fold1_train, dev_fold1_test
  dev_fold1_train_m3: 43 embedding / 104 one-hot features -> dev_fold1_train, dev_fold1_test
  dev_fold1_train_m4: 43 embedding / 104 one-hot features -> dev_fold1_train, dev_fold1_test
  dev_fold1_train_m5: 43 embedding / 104 one-hot features -> dev_fold1_train, dev_fold1_test
  dev_fold2_train_m1: 43 embedding / 104 one-hot features -> dev_fold2

## 4. Report

In [6]:
print("=== rows and feature counts per applied set ===")
print(summary.to_string(index=False))

print("\n=== unseen levels mapped to the reference ===")
if unseen_log:
    ul = pd.DataFrame(unseen_log)
    ul.to_csv(OUT / "unseen_levels.csv", index=False)
    print(ul.to_string(index=False))
    print("\n(a level present in temporal but absent from development maps to the"
          " reference and creates no new column, by design)")
else:
    print("  none")

=== rows and feature counts per applied set ===
        fit_set      applied_to  m  n_rows  n_embedding  n_onehot  unseen_total
dev_fold0_train dev_fold0_train  1   37821           43       104             0
dev_fold0_train  dev_fold0_test  1    9469           43       104             0
dev_fold0_train dev_fold0_train  2   37821           43       104             0
dev_fold0_train  dev_fold0_test  2    9469           43       104             0
dev_fold0_train dev_fold0_train  3   37821           43       104             0
dev_fold0_train  dev_fold0_test  3    9469           43       104             0
dev_fold0_train dev_fold0_train  4   37821           43       104             0
dev_fold0_train  dev_fold0_test  4    9469           43       104             0
dev_fold0_train dev_fold0_train  5   37821           43       104             0
dev_fold0_train  dev_fold0_test  5    9469           43       104             0
dev_fold1_train dev_fold1_train  1   37823           43       104       

In [7]:
# Full spec detail for one representative fit: cardinality per categorical and the
# scaler kind per continuous column.
ref_spec = FeatureSpec.from_json(str(OUT / "spec_dev_full_m1.json"))
print(summarise(ref_spec))

FeatureSpec 'dev_full_m1' fitted on 47,290 rows
  embedding : 16 continuous + 27 categorical codes = 43 columns
  one-hot   : 16 continuous + 88 indicator (sum of levels-1) = 104 columns
  continuous scalers:
    AGE                    standard centre=     53.0680 scale=     13.1239
    AGE_DON                standard centre=     40.3091 scale=     13.2121
    HGT_CM_CALC            standard centre=    169.7833 scale=     10.7671
    HGT_CM_DON_CALC        standard centre=    172.0321 scale=     10.2100
    WGT_KG_CALC            standard centre=     82.0038 scale=     19.2006
    WGT_KG_DON_CALC        standard centre=     84.8131 scale=     21.6211
    CREAT_DON              standard centre=      1.2767 scale=      1.1220
    KDRI_RAO               standard centre=      1.2393 scale=      0.3570
    EPTS_FINAL             standard centre=      0.5016 scale=      0.2982
    DIALYSIS_VINT_DAYS     robust   centre=   1449.5000 scale=   1796.0000
    DAYSWAIT_CHRON_KI      robust   centr

In [8]:
# Total feature count per representation, stated explicitly.
n_cont = len(ref_spec.continuous_order)
n_cat = len(ref_spec.categorical_order)
sum_lvl = sum(len(ref_spec.categorical_levels[c]) - 1
              for c in ref_spec.categorical_order)
print(f"embedding : {n_cont} continuous + {n_cat} categorical codes = "
      f"{ref_spec.n_embedding_features}")
print(f"one-hot   : {n_cont} continuous + {sum_lvl} indicators (sum of levels-1) = "
      f"{ref_spec.n_onehot_features}")
print(f"\nEncoderConfig kwargs for the neural variants: {encoder_config_kwargs(ref_spec)}")

pre_spec = FeatureSpec.from_json(str(OUT / "spec_pretrain_m1.json"))
print(f"\npretrain spec: {pre_spec.n_embedding_features} features "
      f"(masked {sorted(set(PRETRAIN_MASKED))}); "
      f"GLOMERUL retained = {'GLOMERUL' in pre_spec.categorical_order}")

embedding : 16 continuous + 27 categorical codes = 43
one-hot   : 16 continuous + 88 indicators (sum of levels-1) = 104

EncoderConfig kwargs for the neural variants: {'n_continuous': 16, 'categorical_cardinalities': (2, 2, 2, 2, 2, 2, 5, 2, 13, 5, 11, 6, 6, 3, 5, 3, 3, 3, 3, 3, 2, 7, 7, 7, 3, 3, 3)}

pretrain spec: 38 features (masked ['END_CPRA', 'EPTS_FINAL', 'GLOMERUL_COUNT', 'INTERSTITIAL_FIBROSIS', 'VASCULAR_CHANGES']); GLOMERUL retained = True


## 5. Loading the outputs later

```python
d = np.load(OUT / "dev_fold0_train_m1_embedding.npz", allow_pickle=True)
x_cont, x_cat = d["continuous"], d["categorical_codes"]
cardinalities = d["cardinalities"].tolist()          # -> EncoderConfig

d = np.load(OUT / "dev_fold0_train_m1_onehot.npz", allow_pickle=True)
X, names = d["design"], d["columns"].tolist()        # -> classical baselines

y = pd.read_parquet(OUT / "dev_fold0_train_m1_outcomes.parquet")
spec = FeatureSpec.from_json(str(OUT / "spec_dev_fold0_train_m1.json"))
```